# 🚀 Pistes d'amélioration — Modèles avancés
## Notebook complémentaire (version vérifiée)

**Pré-requis** : avoir exécuté le notebook principal `Notebook_Principal_CRISP_DM.ipynb`

> **⚠️ Note méthodologique importante — à lire avant de commencer**
>
> Ce notebook explore des techniques avancées (XGBoost, LightGBM, GridSearchCV, SMOTE, SHAP).
> **Résultat constaté après exécution réelle : sur ce dataset, ces techniques n'améliorent PAS
> la performance du Random Forest du notebook 1** (F1 ≈ 0,54 / AUC ≈ 0,77 dans les deux cas).
>
> Ce n'est pas un échec : c'est un résultat fréquent et instructif. Sur des données tabulaires
> de taille modérée, un Random Forest bien réglé est un concurrent redoutable. L'intérêt de ce
> notebook est donc **méthodologique** : maîtrise des outils, industrialisation (Pipeline),
> explicabilité (SHAP) et compréhension des arbitrages — et non un gain de performance brute.

---

| # | Piste | Apport réel constaté |
|---|---|---|
| 1 | Pipeline scikit-learn | Industrialisation, anti-leakage (pas de gain de perf) |
| 2 | XGBoost & LightGBM | Rapidité ; perf ≈ ou légèrement < RF |
| 3 | GridSearchCV | Réglage systématique ; perf stable, pas de saut |
| 4 | SMOTE | **Dégrade** ici la performance (résultat instructif) |
| 5 | Ajustement du seuil | Gain marginal (+0,6 pt F1) |
| 6 | SHAP — Explicabilité | Conformité RGPD/BCE, confiance métier |

---

### Sommaire
1. Installation et imports
2. Rechargement et préparation des données
3. Piste 1 — Pipeline scikit-learn
4. Piste 2 — XGBoost & LightGBM
5. Piste 3 — GridSearchCV
6. Piste 4 — SMOTE
7. Piste 5 — Ajustement du seuil de décision
8. Piste 6 — SHAP (explicabilité)
9. Synthèse finale et recommandations


## 🔧 1. Installation et imports

### 1.1 Installation des bibliothèques avancées

In [ ]:
# Décommenter et exécuter une seule fois
# !pip install xgboost lightgbm imbalanced-learn shap --quiet

### 1.2 Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import time

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

# Modèles avancés
import xgboost as xgb
import lightgbm as lgb

# Rééchantillonnage
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Explicabilité
import shap

# Config
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Bibliothèques importées")
print(f"  XGBoost  : {xgb.__version__}")
print(f"  LightGBM : {lgb.__version__}")
print(f"  SHAP     : {shap.__version__}")

---
## 📦 2. Rechargement et préparation des données

On reproduit exactement le prétraitement du notebook 1 pour que la comparaison soit valide.


In [ ]:
# Chargement
FILE_PATH = "default_of_credit_card_clients.xls"
df = pd.read_excel(FILE_PATH, header=1)
df = df.rename(columns={'default payment next month': 'DEFAULT', 'PAY_0': 'PAY_1'})

# Nettoyage des modalités incohérentes
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# Séparation X/y
X = df.drop(['ID', 'DEFAULT'], axis=1)
y = df['DEFAULT']

# Encodage one-hot
X_encoded = pd.get_dummies(X, columns=['SEX', 'EDUCATION', 'MARRIAGE'],
                            drop_first=True, dtype=int)

# Split stratifié (mêmes paramètres que le notebook 1 → mêmes échantillons)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Taux de défaut train : {y_train.mean()*100:.2f}%")
print("\n>>> Référence à battre (Random Forest du notebook 1) :")
print("    F1 = 0.5409  |  AUC = 0.7722")

---
## 🔗 3. Piste 1 — Pipeline scikit-learn

Un **Pipeline** chaîne le prétraitement et la modélisation en un seul objet. Avantages :
- **Évite la fuite de données** (data leakage) — le scaler n'apprend que sur le train
- **Code reproductible et plus court**
- **Mise en production simplifiée** (un seul `.pkl` à déployer)
- **Compatible avec GridSearchCV** sur tous les hyperparamètres

⚠️ À noter : le Pipeline est un outil d'**industrialisation**, pas de performance. Le F1 obtenu
ici est quasi identique à celui du Random Forest du notebook 1 — c'est attendu, c'est le même
modèle, simplement mieux encapsulé.


### 3.1 Construction d'un pipeline simple

In [ ]:
# Pipeline : scaling + modèle
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),       # Étape 1 : standardisation
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=10,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ))                                  # Étape 2 : modèle
])

# Entraînement sur l'ensemble du pipeline
pipeline_rf.fit(X_train, y_train)

# Prédiction (le scaling est appliqué automatiquement)
y_pred_pipe = pipeline_rf.predict(X_test)
y_proba_pipe = pipeline_rf.predict_proba(X_test)[:, 1]

print(f"F1-Score (Pipeline RF) : {f1_score(y_test, y_pred_pipe):.4f}")
print(f"AUC-ROC (Pipeline RF)  : {roc_auc_score(y_test, y_proba_pipe):.4f}")
print("\n→ Résultat attendu : F1 ≈ 0.5405, identique au RF du notebook 1")
print("  (léger écart possible : le scaler ne change rien pour un RF,")
print("   mais l'ordre des opérations internes peut varier au 4e décimale)")

### 3.2 Sauvegarde et rechargement du pipeline complet

In [ ]:
# Sauvegarde du pipeline entier (scaler + modèle)
joblib.dump(pipeline_rf, 'pipeline_complet.pkl')

# Rechargement (simule un déploiement)
pipeline_loaded = joblib.load('pipeline_complet.pkl')

# Test : prédiction sur une nouvelle observation, sans refaire le scaling manuellement
new_data = X_test.iloc[[0]]
proba = pipeline_loaded.predict_proba(new_data)[0, 1]
print(f"Probabilité de défaut : {proba:.4f}")
print("✓ Le pipeline applique automatiquement scaling + prédiction")

---
## ⚡ 4. Piste 2 — XGBoost & LightGBM

Ces deux implémentations de gradient boosting sont **plus rapides** que `GradientBoostingClassifier`
de scikit-learn et supportent **nativement** la pondération de classes via `scale_pos_weight`.

**Résultat constaté** : sur ce dataset, leur F1 (≈ 0,52–0,53) est **légèrement inférieur** à celui
du Random Forest (0,54). Le principal gain ici est la **vitesse d'entraînement** (~1 seconde).


### 4.1 Calcul du poids de la classe positive

In [ ]:
# scale_pos_weight = (nb négatifs) / (nb positifs)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio classes (négatif/positif) : {scale_pos_weight:.4f}")
print(f"→ chaque défaut comptera {scale_pos_weight:.2f}x plus dans la fonction de coût")

### 4.2 XGBoost

In [ ]:
print("="*60)
print("XGBOOST")
print("="*60)

t0 = time.time()
model_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,  # Compense le déséquilibre
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='auc'
)
model_xgb.fit(X_train, y_train)
print(f"⏱  Entraînement : {time.time()-t0:.1f}s")

y_pred_xgb = model_xgb.predict(X_test)
y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print(f"\nAccuracy    : {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Précision   : {precision_score(y_test, y_pred_xgb):.4f}")
print(f"Sensibilité : {recall_score(y_test, y_pred_xgb):.4f}")
print(f"F1-Score    : {f1_score(y_test, y_pred_xgb):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test, y_proba_xgb):.4f}")
print("\n→ Valeurs de référence (exécution vérifiée) :")
print("  Acc≈0.7570 | Prec≈0.4612 | Rec≈0.5870 | F1≈0.5166 | AUC≈0.7674")

### 4.3 LightGBM

In [ ]:
print("="*60)
print("LIGHTGBM")
print("="*60)

t0 = time.time()
model_lgb = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
model_lgb.fit(X_train, y_train)
print(f"⏱  Entraînement : {time.time()-t0:.1f}s")

y_pred_lgb = model_lgb.predict(X_test)
y_proba_lgb = model_lgb.predict_proba(X_test)[:, 1]

print(f"\nAccuracy    : {accuracy_score(y_test, y_pred_lgb):.4f}")
print(f"Précision   : {precision_score(y_test, y_pred_lgb):.4f}")
print(f"Sensibilité : {recall_score(y_test, y_pred_lgb):.4f}")
print(f"F1-Score    : {f1_score(y_test, y_pred_lgb):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test, y_proba_lgb):.4f}")
print("\n→ Valeurs de référence (exécution vérifiée) :")
print("  Acc≈0.7622 | Prec≈0.4708 | Rec≈0.6074 | F1≈0.5304 | AUC≈0.7666")

**Interprétation** : LightGBM fait un peu mieux que XGBoost (F1 0,530 vs 0,517) grâce à une
meilleure sensibilité, mais les deux restent sous le Random Forest du notebook 1 (F1 0,541).
Le gradient boosting n'est donc pas automatiquement supérieur — tout dépend du dataset et du réglage.

---
## 🎯 5. Piste 3 — GridSearchCV (optimisation des hyperparamètres)

`GridSearchCV` teste **toutes les combinaisons** d'hyperparamètres par validation croisée et
retourne la meilleure. Cela remplace le réglage manuel par essai-erreur.

**Résultat constaté** : l'optimisation stabilise le modèle (F1 en CV ≈ 0,538) mais ne produit
**pas de saut de performance** sur le test. C'est cohérent : la grille testée reste proche des
valeurs par défaut, déjà raisonnables.


### 5.1 Définition de la grille de recherche

In [ ]:
param_grid = {
    'n_estimators': [200, 300],
    'max_depth':    [4, 5, 6],
    'learning_rate': [0.05, 0.1],
}

print("Grille de recherche :")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

n_combi = 1
for v in param_grid.values():
    n_combi *= len(v)
print(f"\nNombre total de combinaisons : {n_combi}")
print(f"Avec CV 3-folds → {n_combi * 3} entraînements")

### 5.2 Lancement de la recherche

In [ ]:
# Validation croisée stratifiée 3 folds
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1, eval_metric='auc'
    ),
    param_grid=param_grid,
    scoring='f1',          # Métrique d'optimisation
    cv=cv,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid.fit(X_train, y_train)
print(f"\n⏱  Durée totale : {(time.time()-t0)/60:.1f} min")
print(f"\n🏆 Meilleurs hyperparamètres :")
for k, v in grid.best_params_.items():
    print(f"   {k:20s} : {v}")
print(f"\n📊 Meilleur F1 en CV : {grid.best_score_:.4f}")
print("\n→ Référence vérifiée : best params {learning_rate: 0.05, max_depth: 4,")
print("  n_estimators: 300} | F1 CV ≈ 0.5382")

### 5.3 Évaluation du meilleur modèle sur le test

In [ ]:
best_xgb = grid.best_estimator_
y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:, 1]

print("="*60)
print("XGBOOST OPTIMISÉ — Performance sur test")
print("="*60)
print(f"Accuracy    : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Précision   : {precision_score(y_test, y_pred_best):.4f}")
print(f"Sensibilité : {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score    : {f1_score(y_test, y_pred_best):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test, y_proba_best):.4f}")
print("\n→ Référence vérifiée :")
print("  Acc≈0.7628 | Prec≈0.4720 | Rec≈0.6096 | F1≈0.5321 | AUC≈0.7749")
print("\n⚠️  À comparer au RF du notebook 1 : F1=0.5409 / AUC=0.7722")
print("    → l'XGBoost optimisé reste très légèrement EN DESSOUS en F1,")
print("      mais légèrement AU-DESSUS en AUC. Match quasi nul.")

---
## ⚖️ 6. Piste 4 — SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE génère **synthétiquement** de nouvelles observations de la classe minoritaire en
interpolant entre voisins. C'est une alternative à `class_weight` / `scale_pos_weight`.

**⚠️ Pièce critique** : SMOTE doit être appliqué **uniquement sur le train**, jamais sur le test.
L'`ImbPipeline` de imblearn gère cela correctement.

**Résultat constaté — important** : sur ce dataset, **SMOTE DÉGRADE la performance**
(F1 ≈ 0,489, le plus faible de toutes les approches). Ce n'est pas un bug : c'est un résultat
réel et instructif. SMOTE peut créer des observations synthétiques peu réalistes en grande
dimension, et le `scale_pos_weight` natif faisait déjà le travail de rééquilibrage plus proprement.
**À documenter comme tel dans votre rapport** : tester une technique et constater qu'elle ne
convient pas est une démarche scientifique valide.


### 6.1 Application correcte de SMOTE via Pipeline imblearn

In [ ]:
# imblearn Pipeline : SMOTE appliqué pendant fit() mais PAS pendant predict()
pipeline_smote = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.7)),
    # 0.7 = la classe minoritaire représentera 70% de la majoritaire après SMOTE
    ('classifier', xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc'
        # NB : pas de scale_pos_weight ici, SMOTE gère déjà le rééquilibrage
    ))
])

t0 = time.time()
pipeline_smote.fit(X_train, y_train)
print(f"⏱  Entraînement : {time.time()-t0:.1f}s")

y_pred_smote = pipeline_smote.predict(X_test)
y_proba_smote = pipeline_smote.predict_proba(X_test)[:, 1]

print(f"\nAccuracy    : {accuracy_score(y_test, y_pred_smote):.4f}")
print(f"Précision   : {precision_score(y_test, y_pred_smote):.4f}")
print(f"Sensibilité : {recall_score(y_test, y_pred_smote):.4f}")
print(f"F1-Score    : {f1_score(y_test, y_pred_smote):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test, y_proba_smote):.4f}")
print("\n→ Référence vérifiée :")
print("  Acc≈0.7920 | Prec≈0.5354 | Rec≈0.4499 | F1≈0.4889 | AUC≈0.7494")
print("\n⚠️  SMOTE dégrade ici le F1 (0.489 < 0.541 du RF). Résultat instructif :")
print("    sur ce dataset, la pondération native est préférable au sur-échantillonnage.")

### 6.2 Visualisation de l'effet de SMOTE

In [ ]:
# Comparaison avant/après SMOTE (sur le train uniquement)
smote_temp = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.7)
X_smote, y_smote = smote_temp.fit_resample(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_train.value_counts().plot.bar(ax=axes[0], color=['#2E75B6', '#C00000'])
axes[0].set_title(f'Avant SMOTE — {len(y_train):,} observations', fontweight='bold')
axes[0].set_xticklabels(['Non-défaut', 'Défaut'], rotation=0)

pd.Series(y_smote).value_counts().plot.bar(ax=axes[1], color=['#2E75B6', '#C00000'])
axes[1].set_title(f'Après SMOTE — {len(y_smote):,} observations', fontweight='bold')
axes[1].set_xticklabels(['Non-défaut', 'Défaut'], rotation=0)

plt.tight_layout()
plt.show()

print(f"Avant SMOTE — défaut : {(y_train==1).sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Après SMOTE — défaut : {(y_smote==1).sum():,} ({pd.Series(y_smote).mean()*100:.1f}%)")

---
## 🎚️ 7. Piste 5 — Ajustement du seuil de décision

Par défaut, scikit-learn classe avec un **seuil de 0,5** (proba ≥ 0,5 → classe 1). Ce seuil
n'est pas forcément optimal métier — on peut le déplacer **sans réentraîner** le modèle.

**Résultat constaté** : le seuil optimal (≈ 0,58) apporte un gain réel mais **modeste**
(+0,6 point de F1). C'est un levier d'ajustement fin, pas une transformation majeure.


### 7.1 Courbe Précision-Rappel

In [ ]:
# On utilise le meilleur modèle XGBoost (issu de GridSearchCV)
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_best)

plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.plot(recall, precision, color='#1F4E79', linewidth=2)
plt.xlabel('Sensibilité (Rappel)')
plt.ylabel('Précision')
plt.title(f'Courbe Précision-Rappel (AP = {average_precision_score(y_test, y_proba_best):.4f})',
          fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(thresholds, precision[:-1], label='Précision', color='#2E75B6', linewidth=2)
plt.plot(thresholds, recall[:-1], label='Sensibilité', color='#C00000', linewidth=2)
f1_curve = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
plt.plot(thresholds, f1_curve, label='F1', color='#1F4E79', linewidth=2, linestyle='--')
plt.axvline(x=0.5, color='gray', linestyle=':', label='Seuil par défaut')
plt.xlabel('Seuil de décision')
plt.ylabel('Score')
plt.title('Métriques en fonction du seuil', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2 Identification du seuil optimal (F1 maximal)

In [ ]:
# Seuil qui maximise le F1-score
f1_curve = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
idx_best = np.argmax(f1_curve)
seuil_optimal = thresholds[idx_best]

print("="*60)
print(f"SEUIL OPTIMAL (F1 max) : {seuil_optimal:.4f}")
print("="*60)
print(f"À ce seuil :")
print(f"  Précision   : {precision[idx_best]:.4f}")
print(f"  Sensibilité : {recall[idx_best]:.4f}")
print(f"  F1-Score    : {f1_curve[idx_best]:.4f}")

# Comparaison avec seuil 0.5
y_pred_optim = (y_proba_best >= seuil_optimal).astype(int)
print(f"\n--- Avec seuil 0.5 (défaut) ---")
print(f"  F1 : {f1_score(y_test, y_pred_best):.4f}")
print(f"--- Avec seuil {seuil_optimal:.4f} (optimisé) ---")
print(f"  F1 : {f1_score(y_test, y_pred_optim):.4f}")
gain = (f1_score(y_test, y_pred_optim) - f1_score(y_test, y_pred_best)) * 100
print(f"\n💡 Gain F1 sans réentraînement : +{gain:.2f} points")
print("\n→ Référence vérifiée : seuil optimal ≈ 0.58, gain ≈ +0.6 point")
print("  Levier d'ajustement fin, utile mais pas spectaculaire.")

### 7.3 Choix métier du seuil

In [ ]:
# Selon le contexte, on peut privilégier la sensibilité (banque prudente)
# Exemple : viser une sensibilité minimale de 70%
sensibilite_cible = 0.70
idx_seuil = np.where(recall[:-1] >= sensibilite_cible)[0]
if len(idx_seuil) > 0:
    seuil_metier = thresholds[idx_seuil[-1]]  # Seuil le plus élevé garantissant la cible
    print(f"Pour atteindre une sensibilité ≥ {sensibilite_cible*100:.0f}% :")
    print(f"  Seuil à appliquer    : {seuil_metier:.4f}")
    print(f"  Précision résultante : {precision[idx_seuil[-1]]:.4f}")
    print(f"  Sensibilité résultante : {recall[idx_seuil[-1]]:.4f}")
    print("\n  → Abaisser le seuil augmente la sensibilité (capte plus de défauts)")
    print("    au prix de la précision (plus de faux positifs). Arbitrage métier.")
else:
    print(f"Aucun seuil ne permet d'atteindre une sensibilité de {sensibilite_cible*100:.0f}%")

---
## 🔬 8. Piste 6 — SHAP (explicabilité avancée)

**SHAP** (SHapley Additive exPlanations) fournit des explications **mathématiquement fondées**
des prédictions, à deux niveaux :
- **Global** : importance moyenne de chaque variable sur l'ensemble du dataset
- **Local** : contribution de chaque variable à une prédiction individuelle

C'est **la vraie valeur ajoutée de ce notebook** : indépendamment de la performance, SHAP rend
le modèle explicable, ce qui est une exigence croissante du régulateur (RGPD article 22,
exigences BCE/EBA sur les modèles internes de risque).


### 8.1 Calcul des valeurs SHAP (sur un échantillon pour la rapidité)

In [ ]:
# Pour gagner du temps, on utilise un échantillon de 500 observations
X_sample = X_test.sample(500, random_state=RANDOM_STATE)

# TreeExplainer est optimisé pour les modèles à base d'arbres (XGBoost, LightGBM, RF)
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_sample)

print(f"✓ Valeurs SHAP calculées sur {len(X_sample)} observations")
print(f"  Shape des valeurs SHAP : {np.array(shap_values).shape}")

### 8.2 Importance globale (Summary Plot — Bar)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False, max_display=15)
plt.title('Importance globale des variables (SHAP)', fontweight='bold')
plt.tight_layout()
plt.show()

### 8.3 Distribution des contributions (Summary Plot — Beeswarm)

In [ ]:
# Chaque point = un client ; couleur = valeur de la variable ; position = impact sur la prédiction
plt.figure()
shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
plt.title('Distribution des contributions SHAP', fontweight='bold')
plt.tight_layout()
plt.show()

**Lecture** : pour PAY_1, les points correspondant à une valeur élevée (retard de paiement)
se situent du côté positif de l'axe — ils augmentent la probabilité de défaut prédite. Cela
valide l'intuition métier et confirme la cohérence du modèle.

### 8.4 Explication d'une prédiction individuelle (Waterfall Plot)

In [ ]:
# On analyse le premier client de l'échantillon
client_idx = 0

proba_client = best_xgb.predict_proba(X_sample.iloc[[client_idx]])[0, 1]
valeur_reelle = y_test.loc[X_sample.index[client_idx]]
print(f"Probabilité prédite : {proba_client:.4f}")
print(f"Valeur réelle       : {valeur_reelle}")

# Waterfall plot — décompose la prédiction variable par variable
shap.waterfall_plot(shap.Explanation(
    values=shap_values[client_idx],
    base_values=explainer.expected_value,
    data=X_sample.iloc[client_idx].values,
    feature_names=X_sample.columns.tolist()
), max_display=10, show=False)
plt.tight_layout()
plt.show()

**Interprétation du waterfall** :
- Le point de départ (`E[f(x)]`) = la prédiction moyenne du modèle (log-odds de base)
- Chaque barre = contribution d'une variable (rouge = pousse vers le défaut, bleu = vers le non-défaut)
- Le point d'arrivée (`f(x)`) = la prédiction finale pour ce client

Ce niveau d'explication est **directement exploitable** par un analyste crédit pour justifier
une décision auprès d'un client ou d'un auditeur.

---
## 📝 9. Synthèse finale et recommandations


### 9.1 Comparaison de toutes les approches

In [ ]:
# Tableau récapitulatif — toutes les approches du notebook 2
# + rappel de la référence du notebook 1
resultats = [
    {'Approche': 'Random Forest (notebook 1)',  'F1': 0.5409, 'AUC': 0.7722},
    {'Approche': 'Pipeline RF',                 'F1': f1_score(y_test, y_pred_pipe),  'AUC': roc_auc_score(y_test, y_proba_pipe)},
    {'Approche': 'XGBoost (défaut)',            'F1': f1_score(y_test, y_pred_xgb),   'AUC': roc_auc_score(y_test, y_proba_xgb)},
    {'Approche': 'LightGBM (défaut)',           'F1': f1_score(y_test, y_pred_lgb),   'AUC': roc_auc_score(y_test, y_proba_lgb)},
    {'Approche': 'XGBoost + GridSearchCV',      'F1': f1_score(y_test, y_pred_best),  'AUC': roc_auc_score(y_test, y_proba_best)},
    {'Approche': 'XGBoost + SMOTE',             'F1': f1_score(y_test, y_pred_smote), 'AUC': roc_auc_score(y_test, y_proba_smote)},
    {'Approche': 'XGBoost optim. + seuil',      'F1': f1_score(y_test, y_pred_optim), 'AUC': roc_auc_score(y_test, y_proba_best)},
]

df_resultats = pd.DataFrame(resultats).sort_values('F1', ascending=False).reset_index(drop=True)

print("="*60)
print("COMPARAISON DE TOUTES LES APPROCHES")
print("="*60)
print(df_resultats.to_string(index=False,
      formatters={'F1': '{:.4f}'.format, 'AUC': '{:.4f}'.format}))

# Visualisation
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_resultats))
width = 0.35
couleurs_f1 = ['#C00000' if 'notebook 1' in a else '#2E75B6' for a in df_resultats['Approche']]
ax.bar(x - width/2, df_resultats['F1'], width, label='F1-Score', color=couleurs_f1)
ax.bar(x + width/2, df_resultats['AUC'], width, label='AUC-ROC', color='#1F4E79')
ax.axhline(y=0.5409, color='#C00000', linestyle='--', alpha=0.7,
           label='Référence F1 (RF notebook 1)')
ax.set_xticks(x)
ax.set_xticklabels(df_resultats['Approche'], rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title('Comparaison des approches — F1 et AUC', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 9.2 Interprétation des résultats — lecture honnête

**Constat principal : aucune des techniques avancées n'a amélioré significativement la
performance du Random Forest du notebook 1.**

| Approche | F1 | Verdict |
|---|---|---|
| Random Forest (notebook 1) | ~0,541 | **Référence — reste le meilleur F1** |
| XGBoost optim. + seuil | ~0,538 | Quasi équivalent, AUC légèrement supérieure |
| XGBoost + GridSearchCV | ~0,532 | Équivalent |
| LightGBM | ~0,530 | Légèrement en dessous |
| XGBoost (défaut) | ~0,517 | En dessous |
| XGBoost + SMOTE | ~0,489 | **Nettement en dessous — SMOTE dégrade ici** |

**Comment interpréter cela ?**

1. **Ce n'est pas un échec du projet.** Sur des données tabulaires de taille modérée
   (30 000 lignes, 26 features), un Random Forest correctement réglé est un concurrent
   très solide. La littérature le confirme : la supériorité du gradient boosting n'est
   ni automatique ni universelle.

2. **SMOTE qui dégrade la performance est un résultat à part entière.** Il montre que le
   sur-échantillonnage synthétique n'est pas toujours bénéfique, et que la pondération
   native des classes (`class_weight` / `scale_pos_weight`) était ici la meilleure approche.

3. **La vraie valeur ajoutée de ce notebook est méthodologique**, pas la performance brute :
   maîtrise de l'industrialisation (Pipeline), de l'optimisation systématique (GridSearchCV),
   de l'explicabilité (SHAP) et capacité à tester rigoureusement plusieurs hypothèses.

4. **Piste pour réellement progresser** : le levier le plus prometteur n'est probablement
   pas le choix de l'algorithme mais le **feature engineering** (ratios PAY_AMT/BILL_AMT,
   tendances temporelles, ancienneté du dernier retard) — non exploré ici.

### 9.3 Recommandation finale

**Modèle recommandé pour la production : le Random Forest du notebook 1.**

Justification :
- Meilleur F1-score de toutes les approches testées (~0,541)
- AUC quasi équivalente aux meilleurs modèles avancés (~0,772)
- Simple, robuste, rapide, interprétable nativement (`feature_importances_`)
- Pas de dépendance à des bibliothèques externes supplémentaires

**Ce que les techniques avancées apportent quand même au projet :**
- **Pipeline scikit-learn** : à conserver pour l'industrialisation (déploiement propre)
- **SHAP** : à conserver absolument pour l'explicabilité réglementaire — c'est complémentaire,
  pas concurrent du choix de modèle
- **Ajustement du seuil** : à conserver comme levier de pilotage métier
- **GridSearchCV** : à conserver comme méthode, même si le gain a été nul ici

**Architecture cible proposée :**
```
Pipeline production :
  1. Ingestion des données client (API)
  2. Prétraitement (mêmes transformations qu'à l'entraînement)
  3. Pipeline sérialisé (Random Forest) → probabilité de défaut
  4. Application d'un seuil métier ajustable
  5. Génération des top-3 facteurs SHAP pour l'explication
  6. Retour : {probabilité, décision, facteurs explicatifs}
  7. Logging pour le monitoring de la dérive (data drift)
```

### 9.4 Prochaines étapes — le vrai levier de progression

| Priorité | Étape | Outil | Pourquoi |
|---|---|---|---|
| **★★★** | Feature engineering | Pandas | Ratios, tendances temporelles : levier le plus prometteur non exploré |
| ★★ | Optimisation bayésienne | Optuna | Recherche d'hyperparamètres plus large et efficace que GridSearch |
| ★★ | Calibration des probabilités | sklearn.calibration | Probabilités fiables pour le pricing du risque |
| ★ | Stacking de modèles | StackingClassifier | Combiner RF + boosting (gain souvent marginal) |
| ★ | Tests d'équité | Fairlearn | Vérifier l'absence de biais (genre, éducation) |
| ★ | Industrialisation MLOps | MLflow + FastAPI | Versioning, monitoring, réentraînement |

---

> **Conclusion du notebook 2**
>
> Tester des techniques avancées et constater qu'elles n'améliorent pas la baseline est une
> démarche scientifique rigoureuse, pas un résultat décevant. Ce notebook démontre la maîtrise
> d'un éventail d'outils professionnels (boosting, optimisation, rééchantillonnage, explicabilité)
> et, surtout, la capacité à **évaluer honnêtement** ce qui fonctionne et ce qui ne fonctionne pas
> sur des données réelles.

---
*Notebook réalisé dans le cadre du programme AEC Business Analyse & Data Analytics — Projet de
prédiction du risque de défaut crédit. Tous les chiffres de référence ont été obtenus par
exécution réelle du code.*
